# ECAPA-TDNN: VLSP Public Test на Kaggle

Готовая версия для обновлённого проекта. Обучение не запускается.
Dataset `skylak3/ecapa-tdnn` и датасет VLSP.
Accelerator GPU T4 x2 и Internet On для установки зависимостей.


In [ ]:
from pathlib import Path

SRC = Path('/kaggle/input/datasets/skylak3/ecapa-tdnn')
DST = Path('/kaggle/working/ECAPA')
VLSP_BASE = Path('/kaggle/input/datasets/dungnasa10/vietnamese-speaker-verification/data/data')
TRIALS = VLSP_BASE / 'metadata/test/labels/public_test_pairs_with_label.txt'
AUDIO_ROOT = VLSP_BASE / 'test/sv_vlsp_2021/public_test/competition_public_test'
MODEL_NAME = 'model_0082.model'

# Kaggle may mount a dataset under a shorter path. Only use an existing observed mount.
if not SRC.is_dir():
    alternative = Path('/kaggle/input/ecapa-tdnn')
    if alternative.is_dir():
        SRC = alternative
DATASET_ROOT = SRC
if (SRC / 'ecapa-tdnn-kaggle/train/run.py').is_file():
    SRC = SRC / 'ecapa-tdnn-kaggle'
print('Code dataset:', SRC)
print('Trials:', TRIALS)
print('Audio:', AUDIO_ROOT)


In [ ]:
import shutil
import os

if not (SRC / 'train/run.py').is_file():
    raise FileNotFoundError(f'{SRC}/train/run.py missing. Attach the updated dataset with extracted project folders.')
DST.mkdir(parents=True, exist_ok=True)
for name in ['models','train','eval','diarize','utils','experiments','configs','tests']:
    shutil.copytree(SRC / name, DST / name, dirs_exist_ok=True)
for name in ['requirements.txt','requirements-dev.txt','pytest.ini','README.md','IMPLEMENTATION_NOTES.md']:
    shutil.copy2(SRC / name, DST / name)
checkpoint_dir = DST / 'exps/exp1/model'
checkpoint_dir.mkdir(parents=True, exist_ok=True)
for checkpoint in sorted(DATASET_ROOT.rglob('model_*.model')):
    target = checkpoint_dir / checkpoint.name
    if not target.exists():
        shutil.copy2(checkpoint, target)
        print('Copied checkpoint:', target.name)
    else:
        print('Kept existing checkpoint:', target.name)
os.chdir(DST)
print('Working directory:', DST)


In [ ]:
import sys
import subprocess
import importlib.metadata
import torch
import torchaudio

print('torch:', torch.__version__, 'torchaudio:', torchaudio.__version__)
print('CUDA available:', torch.cuda.is_available(), 'GPUs:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))
# Keep the installed Kaggle torch/torchaudio builds instead of replacing CUDA packages.
constraints = DST / 'kaggle_torch_constraints.txt'
constraints.write_text(''.join(f'{name}=={importlib.metadata.version(name)}\n' for name in ['torch','torchaudio']))
subprocess.check_call([sys.executable,'-m','pip','install','-r','requirements-dev.txt','-c',str(constraints)])


In [ ]:
subprocess.check_call([sys.executable,'-m','pytest','-q'])
MODEL = DST / 'exps/exp1/model' / MODEL_NAME
for path in [MODEL, TRIALS]:
    if not path.is_file():
        raise FileNotFoundError(path)
if not AUDIO_ROOT.is_dir():
    raise FileNotFoundError(AUDIO_ROOT)
print('Ready:', MODEL)


## Полный VLSP Public Test protocol

Формат из исходного notebook: `wav1,wav2,label`, через запятую, без заголовка.
Все пары обрабатываются целиком. Ошибки не пропускаются.
Результат сохраняется только после успешной оценки; значения не предзаполнены.


In [ ]:
command = [sys.executable,'eval/eer.py',
    '--model_path',str(MODEL), '--dataset','vlsp', '--trials_format','vlsp',
    '--trials',str(TRIALS), '--audio_root',str(AUDIO_ROOT), '--split','public_test',
    '--device','auto', '--seed','42', '--output',f'results/vlsp_public_test_{MODEL.stem}.json']
print(' '.join(command))
subprocess.check_call(command)


In [ ]:
import json
from IPython.display import FileLink, display
result_path = DST / f'results/vlsp_public_test_{MODEL.stem}.json'
result = json.loads(result_path.read_text())
for key in ['dataset','model','pairs','positive_pairs','negative_pairs','EER','minDCF','threshold']:
    print(f'{key}: {result[key]}')
display(FileLink(str(result_path)))


## Сравнение checkpoints — отдельный запуск по необходимости

Следующий код по умолчанию выключен. Установите `RUN_CHECKPOINT_COMPARISON = True`,
если нужные модели уже скопированы и вы хотите оценить каждый checkpoint.
Их список сохраняется; автоматического удаления нет.


In [ ]:
RUN_CHECKPOINT_COMPARISON = False
if RUN_CHECKPOINT_COMPARISON:
    subprocess.check_call([sys.executable,'experiments/eval_checkpoints.py',
        '--model_dir',str(MODEL.parent), '--dataset','vlsp', '--trials_format','vlsp',
        '--trials',str(TRIALS), '--audio_root',str(AUDIO_ROOT), '--split','public_test',
        '--every','1', '--device','auto', '--seed','42', '--output','results/eer_by_epoch.csv'])


## Обучение и DER

Для обучения используйте команды `train/run.py --dataset dolly|vlsp` из README
и отдельный `save_path` для каждой стадии. Старые ячейки замены импортов не нужны.
Для DER необходимы многоспикерные записи с reference RTTM. Verification pairs
не заменяют diarization corpus. Обучение и ablations здесь автоматически не запускаются.
